# V5 Drive Audit — Pre-flight check for Colab runs

**Purpose:** Verify your Drive has everything needed for Task 3.2 training AND that the code on Drive matches the latest GitHub master. Run this BEFORE `v5_training.ipynb` so you don't waste 1-3 hours of Colab compute on stale code.

**What it checks:**
1. Expected file tree is present (training package + features package + parquets + notebooks)
2. Each code file's SHA-256 hash matches GitHub master (detects stale uploads)
3. Feature parquets are readable and have expected row counts
4. Python imports resolve end-to-end

**Runtime:** ~30 seconds. No compute — just file listings + small HTTP fetches from GitHub.

## Step 1 — Mount Drive + set paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/SportsAnalyzer')
GITHUB_REPO = 'jtemblador/SportsAnalyzer'  # adjust if forked
GITHUB_BRANCH = 'master'

assert DRIVE_ROOT.exists(), f'Drive root missing: {DRIVE_ROOT}'
print(f'Drive root: {DRIVE_ROOT}')
print(f'GitHub ref: {GITHUB_REPO}@{GITHUB_BRANCH}')

Mounted at /content/drive
Drive root: /content/drive/MyDrive/SportsAnalyzer
GitHub ref: jtemblador/SportsAnalyzer@master


## Step 2 — Expected file tree

Every file below should be present. Missing = upload it. Wrong hash = stale, re-upload from local.

In [ ]:
# Code files to hash-compare against GitHub (path relative to repo root)
CODE_FILES = [
    # V5 training package (Task 3.2 — uploaded for HANDOFF #2)
    'src/nfl/training/v5/__init__.py',
    'src/nfl/training/v5/config.py',
    'src/nfl/training/v5/data.py',
    'src/nfl/training/v5/models.py',
    'src/nfl/training/v5/walkforward.py',
    'src/nfl/training/v5/train.py',
    'src/nfl/training/v5/ablation.py',                  # NEW — Task 3.2b ablation entry
    # V5 features package (Task 3.1 + 3.1.5 — uploaded for HANDOFF #1.5)
    'src/nfl/features/v5/__init__.py',
    'src/nfl/features/v5/config.py',
    'src/nfl/features/v5/master_table.py',
    'src/nfl/features/v5/rolling.py',
    'src/nfl/features/v5/context.py',
    'src/nfl/features/v5/usage.py',
    'src/nfl/features/v5/advanced.py',
    'src/nfl/features/v5/dst.py',
    'src/nfl/features/v5/utils.py',
    'src/nfl/features/v5/engineer.py',
]

# Notebooks to hash-compare (these are typically uploaded separately OR opened from GitHub directly)
NOTEBOOK_FILES = [
    'colab/v5_feature_engineering.ipynb',
    'colab/v5_training.ipynb',
    'colab/v5_ablation.ipynb',                         # NEW — Task 3.2b runner
]

# Data parquets to verify existence + row count (HANDOFF #1.5 output)
EXPECTED_SEASONS = list(range(2018, 2026))  # 8 seasons
PARQUET_EXPECTATIONS = {
    # path_pattern: (min_rows, max_rows) per season
    'output/features/v5/features_{season}.parquet': (5500, 7200),       # player: ~6K-7K per season
    'output/features/v5/features_dst_{season}.parquet': (500, 600),     # DST: ~544 per season
}

## Step 3 — Hash-compare code against GitHub master

Fetches raw file content from GitHub and compares SHA-256 against Drive copy. Fast because Python files are small. If your Drive copy's hash differs from GitHub master, you have a stale upload — re-copy the file from your local repo.

In [4]:
import hashlib, urllib.request, urllib.error

def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()[:12]  # short hash for display

def fetch_github_raw(path: str) -> bytes | None:
    url = f'https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_BRANCH}/{path}'
    try:
        with urllib.request.urlopen(url, timeout=10) as r:
            return r.read()
    except urllib.error.HTTPError:
        return None
    except Exception as e:
        print(f'  [GitHub error] {path}: {e}')
        return None

def audit_file(rel_path: str, label: str) -> dict:
    drive_path = DRIVE_ROOT / rel_path
    result = {'path': rel_path, 'label': label, 'status': 'unknown'}
    if not drive_path.exists():
        result['status'] = 'MISSING'
        return result
    drive_bytes = drive_path.read_bytes()
    result['drive_hash'] = sha256_bytes(drive_bytes)
    result['size_kb'] = len(drive_bytes) / 1024
    gh_bytes = fetch_github_raw(rel_path)
    if gh_bytes is None:
        result['status'] = 'GITHUB_UNREACHABLE'
        result['github_hash'] = '?'
    else:
        result['github_hash'] = sha256_bytes(gh_bytes)
        result['status'] = 'MATCH' if result['drive_hash'] == result['github_hash'] else 'STALE'
    return result

print('Auditing code files (this takes ~15s for GitHub fetches)...\n')
code_results = [audit_file(p, 'code') for p in CODE_FILES]
notebook_results = [audit_file(p, 'notebook') for p in NOTEBOOK_FILES]
all_file_results = code_results + notebook_results

print(f'\n{"STATUS":<22} {"SIZE KB":>8}  {"DRIVE":<14} {"GITHUB":<14}  PATH')
print('-' * 120)
for r in all_file_results:
    status = r['status']
    size = f"{r.get('size_kb', 0):.1f}" if 'size_kb' in r else '-'
    d_hash = r.get('drive_hash', '-')
    gh_hash = r.get('github_hash', '-')
    marker = '\u2713' if status == 'MATCH' else ('\u2717' if status in ('STALE', 'MISSING') else '?')
    print(f'{marker} {status:<20} {size:>8}  {d_hash:<14} {gh_hash:<14}  {r["path"]}')

Auditing code files (this takes ~15s for GitHub fetches)...


STATUS                  SIZE KB  DRIVE          GITHUB          PATH
------------------------------------------------------------------------------------------------------------------------
✓ MATCH                     0.5  309df23e8b85   309df23e8b85    src/nfl/training/v5/__init__.py
✓ MATCH                     3.4  3f4770c0811d   3f4770c0811d    src/nfl/training/v5/config.py
✓ MATCH                     9.9  2096e3ea8d83   2096e3ea8d83    src/nfl/training/v5/data.py
✓ MATCH                    10.4  24813142fab9   24813142fab9    src/nfl/training/v5/models.py
✓ MATCH                     6.3  cf80674c8ad7   cf80674c8ad7    src/nfl/training/v5/walkforward.py
✓ MATCH                     9.0  ea9efd61d6fc   ea9efd61d6fc    src/nfl/training/v5/train.py
✓ MATCH                     0.2  ed2648ced281   ed2648ced281    src/nfl/features/v5/__init__.py
✓ MATCH                     5.0  0a3f05628ff4   0a3f05628ff4    src/nfl/features/v5/

## Step 4 — Feature parquet audit (presence + row counts)

In [5]:
import pandas as pd

print('Auditing feature parquets...\n')
parquet_issues = []
for pattern, (min_rows, max_rows) in PARQUET_EXPECTATIONS.items():
    print(f'\n{pattern}')
    for season in EXPECTED_SEASONS:
        path = DRIVE_ROOT / pattern.format(season=season)
        if not path.exists():
            print(f'  \u2717 MISSING: {path.name}')
            parquet_issues.append(f'missing: {path.name}')
            continue
        try:
            df = pd.read_parquet(path)
            n = len(df)
            cols = len(df.columns)
            marker = '\u2713' if min_rows <= n <= max_rows else '\u26a0'
            msg = f'  {marker} {path.name}: {n:,} rows \u00d7 {cols} cols'
            if not (min_rows <= n <= max_rows):
                msg += f'  (expected {min_rows}-{max_rows} rows)'
                parquet_issues.append(f'row count anomaly: {path.name} has {n}')
            print(msg)
        except Exception as e:
            print(f'  \u2717 READ ERROR: {path.name} \u2014 {e}')
            parquet_issues.append(f'read error: {path.name}')

Auditing feature parquets...


output/features/v5/features_{season}.parquet
  ✓ features_2018.parquet: 6,114 rows × 237 cols
  ✓ features_2019.parquet: 6,166 rows × 237 cols
  ✓ features_2020.parquet: 6,338 rows × 237 cols
  ✓ features_2021.parquet: 6,690 rows × 237 cols
  ✓ features_2022.parquet: 6,662 rows × 238 cols
  ✓ features_2023.parquet: 6,650 rows × 238 cols
  ✓ features_2024.parquet: 6,692 rows × 238 cols
  ✓ features_2025.parquet: 6,895 rows × 238 cols

output/features/v5/features_dst_{season}.parquet
  ✓ features_dst_2018.parquet: 512 rows × 56 cols
  ✓ features_dst_2019.parquet: 512 rows × 56 cols
  ✓ features_dst_2020.parquet: 512 rows × 56 cols
  ✓ features_dst_2021.parquet: 544 rows × 56 cols
  ✓ features_dst_2022.parquet: 542 rows × 56 cols
  ✓ features_dst_2023.parquet: 544 rows × 56 cols
  ✓ features_dst_2024.parquet: 544 rows × 56 cols
  ✓ features_dst_2025.parquet: 544 rows × 56 cols


## Step 5 — Import smoke test

In [ ]:
import sys
if str(DRIVE_ROOT) not in sys.path:
    sys.path.insert(0, str(DRIVE_ROOT))

# Ensure __init__.py exists at each intermediate package level (Drive lacks them by default)
for init_path in [
    DRIVE_ROOT / 'src/__init__.py',
    DRIVE_ROOT / 'src/nfl/__init__.py',
    DRIVE_ROOT / 'src/nfl/training/__init__.py',
]:
    init_path.touch(exist_ok=True)

import_errors = []
try:
    from src.nfl.training.v5 import StatPredictor, POBModel, train_all  # noqa: F401
    from src.nfl.training.v5.config import POSITION_ALGORITHMS, COUNT_STATS  # noqa: F401
    from src.nfl.training.v5.data import fill_features, NEUTRAL_FILLS  # noqa: F401
    from src.nfl.features.v5 import build_features, build_dst_features  # noqa: F401
    # Task 3.2b ablation imports + opp_offense group sanity check
    from src.nfl.training.v5.ablation import run_all_ablations, ABLATION_GROUPS  # noqa: F401
    from src.nfl.features.v5.config import FEATURE_GROUP_PREFIXES
    assert 'opp_offense' in FEATURE_GROUP_PREFIXES, \
        'config.py is stale — missing opp_offense group (Task 3.2b requirement)'
    print('\u2713 All training + feature + ablation imports resolve')
    print(f'  FEATURE_GROUP_PREFIXES: {list(FEATURE_GROUP_PREFIXES)}')
except Exception as e:
    print(f'\u2717 Import failed: {type(e).__name__}: {e}')
    import_errors.append(f'{type(e).__name__}: {e}')

## Step 6 — Final summary + action list

In [7]:
stale = [r for r in all_file_results if r['status'] == 'STALE']
missing_code = [r for r in all_file_results if r['status'] == 'MISSING']
unreachable = [r for r in all_file_results if r['status'] == 'GITHUB_UNREACHABLE']
matching = [r for r in all_file_results if r['status'] == 'MATCH']

print('=' * 60)
print('AUDIT SUMMARY')
print('=' * 60)
print(f'  Code/notebook files matching GitHub:     {len(matching)} / {len(all_file_results)}')
print(f'  Stale (re-upload needed):                {len(stale)}')
print(f'  Missing from Drive:                      {len(missing_code)}')
print(f'  GitHub unreachable (inconclusive):       {len(unreachable)}')
print(f'  Parquet issues:                          {len(parquet_issues)}')
print(f'  Import errors:                           {len(import_errors)}')

all_good = not (stale or missing_code or parquet_issues or import_errors)

if all_good:
    print('\n\u2713 READY to run v5_training.ipynb')
else:
    print('\n\u2717 ACTION REQUIRED before running training:')
    if stale:
        print('\n  STALE FILES (re-upload from local repo):')
        for r in stale:
            print(f'    - {r["path"]}  (Drive {r["drive_hash"]} vs GitHub {r["github_hash"]})')
    if missing_code:
        print('\n  MISSING FILES (upload these):')
        for r in missing_code:
            print(f'    - {r["path"]}')
    if parquet_issues:
        print('\n  PARQUET ISSUES (re-run v5_feature_engineering.ipynb):')
        for msg in parquet_issues:
            print(f'    - {msg}')
    if import_errors:
        print('\n  IMPORT ERRORS (missing __init__.py or broken module):')
        for msg in import_errors:
            print(f'    - {msg}')
    if unreachable:
        print('\n  INCONCLUSIVE (GitHub fetch failed \u2014 manually verify these):')
        for r in unreachable:
            print(f'    - {r["path"]}')

AUDIT SUMMARY
  Code/notebook files matching GitHub:     16 / 18
  Stale (re-upload needed):                0
  Missing from Drive:                      1
  GitHub unreachable (inconclusive):       1
  Parquet issues:                          0
  Import errors:                           0

✗ ACTION REQUIRED before running training:

  MISSING FILES (upload these):
    - colab/v5_training.ipynb

  INCONCLUSIVE (GitHub fetch failed — manually verify these):
    - colab/v5_feature_engineering.ipynb
